In [2]:
from importlib import reload
from IPython.core.interactiveshell import InteractiveShell
%load_ext autoreload
InteractiveShell.ast_node_interactivity = "all"
import logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)

In [3]:
import numpy as np
import pandas as pd
import os
import sys
import matplotlib.pyplot as plt

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

2024-10-30 12:18:44,247 - numexpr.utils - INFO - Note: NumExpr detected 32 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
2024-10-30 12:18:44,249 - numexpr.utils - INFO - NumExpr defaulting to 8 threads.
/cmnfs/home/z.xiao/miniconda3/envs/sbs/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Load data

In [60]:
## IM_ref
swaps_config_path = "/cmnfs/proj/ORIGINS/SWAPS_exp/short_gradient/30min_3to45_7R_120min_lib_im_ref_20241002_165602_293498/config_20241003_074426_198157.yaml"
ps_dir = "exp_20241003_083433_946837"
from utils.config import get_cfg_defaults
from utils.singleton_swaps_optimization import swaps_optimization_cfg

cfg = get_cfg_defaults(swaps_optimization_cfg)
cfg.merge_from_file(swaps_config_path)
cfg.PEAK_SELECTION.merge_from_file(
    os.path.join(
        cfg.RESULT_PATH, "peak_selection", ps_dir, "updated_peak_selection_config.yaml"
    )
)
maxquant_result_ref = pd.read_pickle(cfg.DICT_PICKLE_PATH)

mobility_values_df = pd.read_csv(os.path.join(cfg.RESULT_PATH, "mobility_values.csv"))
ms1scans = pd.read_csv(os.path.join(cfg.RESULT_PATH, "ms1scans.csv"))
pept_act_sum_ps_full_tdc = pd.read_csv(
    os.path.join(
        cfg.RESULT_PATH,
        "peak_selection",
        ps_dir,
        "pept_act_sum_ps_full_tdc_fdr_thres.csv",
    )
)
test_pred_df = pd.read_csv(
    os.path.join(
        cfg.RESULT_PATH, "peak_selection", ps_dir, "results", "test_pred_df.csv"
    )
)
# test_pred_df = pd.merge(test_pred_df, maxquant_result_ref, on=["mz_rank", "Decoy"])
test_pred_df = pd.merge(test_pred_df, maxquant_result_ref, on=["mz_rank", "Decoy"])
save_dir = "/cmnfs/proj/ORIGINS/SWAPS_exp/SWAPS_paper_figures/fig3_FDR/ref_vs_pred_im/"
dataset_name = "im_ref"

In [72]:
## IM pred
swaps_config_path = "/cmnfs/proj/ORIGINS/SWAPS_exp/short_gradient/30min_3to45_7R_120min_lib_im_pred_20241004_074414_299434/config_20241004_074414_299434.yaml"
ps_dir = "exp_20241004_093657_807920"

cfg = get_cfg_defaults(swaps_optimization_cfg)
cfg.merge_from_file(swaps_config_path)
cfg.PEAK_SELECTION.merge_from_file(
    os.path.join(
        cfg.RESULT_PATH, "peak_selection", ps_dir, "updated_peak_selection_config.yaml"
    )
)
maxquant_result_ref = pd.read_pickle(cfg.DICT_PICKLE_PATH)

mobility_values_df = pd.read_csv(os.path.join(cfg.RESULT_PATH, "mobility_values.csv"))
ms1scans = pd.read_csv(os.path.join(cfg.RESULT_PATH, "ms1scans.csv"))
pept_act_sum_ps_full_tdc = pd.read_csv(
    os.path.join(
        cfg.RESULT_PATH,
        "peak_selection",
        ps_dir,
        "pept_act_sum_ps_full_tdc_fdr_thres.csv",
    )
)
test_pred_df = pd.read_csv(
    os.path.join(
        cfg.RESULT_PATH, "peak_selection", ps_dir, "results", "test_pred_df.csv"
    )
)
# test_pred_df = pd.merge(test_pred_df, maxquant_result_ref, on=["mz_rank", "Decoy"])
test_pred_df = pd.merge(test_pred_df, maxquant_result_ref, on=["mz_rank", "Decoy"])
save_dir = "/cmnfs/proj/ORIGINS/SWAPS_exp/SWAPS_paper_figures/fig3_FDR/ref_vs_pred_im/"
dataset_name = "im_pred"

# Peak selection and scoring model performance
Use only test_pred_df

In [73]:
from matplotlib.colors import ListedColormap

custom_cmap = ListedColormap(["#FF5733", "#33FF57", "#3357FF"])  # Custom colors
custom_labels = ["Targets", "Decoys"]  # Custom label names

## Plot

In [74]:
test_pred_df_full = pd.merge(
    left=test_pred_df,
    right=maxquant_result_ref[["mz_rank", "Decoy"]],
    on=["mz_rank", "Decoy"],
    how="left",
)
test_pred_df_full["Data"] = "Target"
test_pred_df_full.loc[test_pred_df_full["Decoy"], "Data"] = "Decoy"

## Test set targets

In [75]:
test_pred_df_targets = test_pred_df_full.loc[~test_pred_df_full["Decoy"]]
test_pred_df_targets["Pass Intensity Filter"] = (
    test_pred_df_targets["log_sum_intensity"] >= 2
)
test_pred_df_targets["Pass Conf. Score Filter"] = (
    test_pred_df_targets["target_decoy_score"] >= 0.2
)
test_pred_df_targets["Pass Both Filter"] = (
    test_pred_df_targets["Pass Intensity Filter"]
    & test_pred_df_targets["Pass Conf. Score Filter"]
)

/tmp/ipykernel_843811/3333441363.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_pred_df_targets["Pass Intensity Filter"] = (
/tmp/ipykernel_843811/3333441363.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_pred_df_targets["Pass Conf. Score Filter"] = (
/tmp/ipykernel_843811/3333441363.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas

In [76]:
%autoreload 2
from peak_detection_2d.utils import plot_per_image_metric_distr

plot_per_image_metric_distr(
    loss_array=test_pred_df_targets,
    metric_name="per_image_weighted_iou_metric",
    save_dir=save_dir,
    hue="Pass Intensity Filter",
    title ="Test Set Weighted IoU (Pred IM)",
    xlabel="Weighted IoU",
    show_quantiles=[50],
    palette={False: "#d8a6a6", True: "#a00000"},
    hue_order=[False, True],
    dataset_name=dataset_name
)

/cmnfs/home/z.xiao/.local/lib/python3.10/site-packages/seaborn/_base.py:948: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)
/cmnfs/home/z.xiao/.local/lib/python3.10/site-packages/seaborn/_base.py:948: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)
/cmnfs/home/z.xiao/.local/lib/python3.10/site-packages/seaborn/_base.py:948: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)


50%: 0.82


2024-10-30 13:52:35,111 - utils.plot - INFO - Save plot at /cmnfs/proj/ORIGINS/SWAPS_exp/SWAPS_paper_figures/fig3_FDR/ref_vs_pred_im/PS_model_test_per_image_weighted_iou_metric_distribution_im_pred.png
2024-10-30 13:52:35,406 - utils.plot - INFO - Save plot at /cmnfs/proj/ORIGINS/SWAPS_exp/SWAPS_paper_figures/fig3_FDR/ref_vs_pred_im/PS_model_test_per_image_weighted_iou_metric_distribution_im_pred.svg


In [77]:
from peak_detection_2d.utils import plot_roc_auc

plot_roc_auc(
    pred_df=test_pred_df,
    save_dir=save_dir,
    dataset_name=dataset_name,
    title="Test Set ROC AUC (Pred IM)",
)

2024-10-30 13:52:42,764 - utils.plot - INFO - Save plot at /cmnfs/proj/ORIGINS/SWAPS_exp/SWAPS_paper_figures/fig3_FDR/ref_vs_pred_im/roc_auc_im_pred.png
2024-10-30 13:52:42,954 - utils.plot - INFO - Save plot at /cmnfs/proj/ORIGINS/SWAPS_exp/SWAPS_paper_figures/fig3_FDR/ref_vs_pred_im/roc_auc_im_pred.svg


0.7199242651558383